# 45 · Many-draw specimen nulls and an AKI positive control for the pathway-screen specificity claim (R2)

Paper section R2 says that with two specimens per group, **average-level** competitive pathway
screens report as many pathways for balanced specimen relabelings as for the species split, while
**position-dependent** screens do not (notebooks 31 and 37). The internal referee (M6, D5, D6)
objected that this rests on two null draws, the same two draws that chose the strategy, test,
coordinate and criteria, and that the split-plot df argument does not by itself predict it.

This notebook asks the question again with many independent null draws and one positive control:

| Part | Question | Data |
|---|---|---|
| D5 · same-species nulls | How many pathways does each strategy call when two random pairs of donors of one species are compared? | Census mouse (12 male, 12 female donors), Lake/KPMP human cortex (7), Census human cortex (6) |
| D5 · cross-species draws | In many 2 human + 2 mouse designs, how do species calls compare with relabeled calls, and with an all-donor reference? | Lake human + Census male mouse |
| Anchor | Where do our own cohort's relabelings fall, with identical machinery? | Notebook 37 structures |
| Mechanism | Are specimen offsets more coherent within pathways than specimen-by-segment deviations? | All pools |
| D6 · positive control | Does a position-dependent screen find AKI injury concentrated in S3, and how does it behave under relabeling? | Our 2 control + 2 AKI mice |

The public atlases label segments but have no coordinate, so the screens run at **segment
resolution**: a group offset (`T_level`) against group × segment steps (`T_steps`). The protocol,
priors and decision rules were fixed before any statistic was computed (workstream 5 plan,
reproduced in section 1).

## 1 · Folders and pre-specified rules

| Rule | Value |
|---|---|
| Genes | notebook 12/37 universe; per pool, measured and detected in ≥ 2% of units (equal-donor mean) |
| Normalization | log1p(count / L × 10⁴), L = counts over every accepted ortholog measured in the pool |
| Pathways | notebook 37's 1,513 sets restricted to the pool's tested genes, 10–300 members |
| Unit-level statistics | notebook 37's weights and specimen contrasts; segment intercepts as shared shape; `T_level` = + group offset; `T_steps` = + group × segment beyond the offset; true factor's segment shape as nuisance in relabelings |
| Pseudobulk statistics | voom-style moderated t of the group coefficient, whole PT and per segment (true factor as covariate in relabelings); declared substitute for DESeq2 |
| Pathway tests | matched rank-AUC with the exact matched-null mean/SD; joint = z ÷ √VIF (pool VIF from within-cell residual correlations); signed GSEA (gseapy multilevel, weight 1) |
| Calls | BH q ≤ 0.05 within partition and strategy (positive effect for unsigned F); segment pseudobulk: BH over pathway × segment, called if any segment passes |
| Primary strategies | O1 whole-PT pseudobulk + GSEA · O2 S1/S2/S3 pseudobulk + GSEA · O3 `T_level` joint · P1 `T_steps` joint |
| Draws (seed 45) | N1/N2 Census mouse male/female: 100 random 4-donor sets × 3 splits · N3 Lake: all 105 · N4 Census human: all 45 · N5 cross-species: 100 random 2 + 2 draws × 3 partitions |
| R2 general | N5 median NCDR(P1) < 0.25 with ≥ 80% of draws meeting notebook 31's rule, O1 median NCDR ≥ 0.5, **and** P1's 90th-percentile null calls ≤ 5% of tested in ≥ 3 of 4 same-species pools |
| R2 cohort-specific | N5 median NCDR(P1) ≥ 0.5, or P1's 90th percentile > 5% in ≥ 2 of 4 same-species pools |
| D6 pass | a positional screen calls ≥ 3 injury Hallmarks (or ranks them above others, Mann–Whitney p ≤ 0.01), relabel calls ≤ 5% of condition calls, and ≥ 2/3 of tested injury markers rise most in S3 |

In [ ]:
import argparse
import json
import os
import sys
from itertools import combinations
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
parser.add_argument('--workers', type=int, default=12)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
WORKERS = int(os.environ.get('PSEUDOSPACE_WORKERS', args.workers))
upstream13 = results_root / 'minimal_pt_scfates'
upstream37 = results_root / 'pt_pathway_final' / 'scfates' / 'tables'
mouse_v5 = results_root / 'mouse_only_v5'
external = data_root / 'external'
output = results_root / 'pt_revision_method'
figures, tables = output / 'figures', output / 'tables'
for folder in (figures, tables):
    folder.mkdir(parents=True, exist_ok=True)
census_cells = {s: external / 'census_pt_cells' / f'{s}_pt_cells.h5ad' for s in ('mouse', 'human')}
lake_file = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv',
            upstream37 / 'gene_level_statistics.csv', upstream37 / 'matched_and_joint_tests_all_partitions.csv',
            upstream37 / 'table_S1_strategy_specificity.csv',
            mouse_v5 / 'all_mouse_tubules_scanpy_dpt.h5ad', mouse_v5 / 'all_mouse_tubules_harmony_pass1.h5ad',
            lake_file, *census_cells.values()]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (Census cells: analysis/scripts/fetch_census_pt_cells.py)')

NOTEBOOK_LOGIC_VERSION = '45.specimen_null.1'  # Bump when a cached calculation changes.
SEED = 45
ALPHA = .05
DETECTION = .02
MIN_UNITS = 50              # nuclei per donor and segment (notebook 37's external rule)
N_SETS = 100                # random 4-donor sets per large same-species pool
N_CROSS = 100               # random 2 + 2 cross-species draws
GSEA_SEED = 12
NULL_RATE = .05             # notebook 31's null-rate condition
INJURY_MARKERS = ['Havcr1', 'Lcn2', 'Krt20', 'Krt8', 'Krt18', 'Spp1', 'Cd44', 'Vcam1', 'Sox9', 'Cdkn1a',
                  'Clu', 'Mt1', 'Mt2', 'Atf3', 'Hmox1', 'Timp1', 'Sprr1a', 'Tnfrsf12a']
INJURY_HALLMARKS = ['TNF-alpha Signaling via NF-kB', 'p53 Pathway', 'Apoptosis', 'Hypoxia', 'Inflammatory Response',
                    'IL-6/JAK/STAT3 Signaling', 'Interferon Gamma Response', 'Epithelial Mesenchymal Transition',
                    'Unfolded Protein Response', 'mTORC1 Signaling', 'TGF-beta Signaling', 'Fatty Acid Metabolism',
                    'Oxidative Phosphorylation']
print('Results folder:', output)

## 2 · Gene universe and pathways (notebook 37's definitions)

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
from pseudospace.pathways import build_pathway_membership

pd.set_option('display.width', 220, 'display.max_columns', 30, 'display.max_colwidth', 60)
universe_genes = pd.read_csv(results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv', index_col=0).index.tolist()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
libraries = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
coverage = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{library}.json').read_text()),
    universe_genes, ortholog_map=orthologs, library_name=library, tested=universe_genes, min_genes=1)
    for library in libraries], ignore_index=True)
coverage['pathway_id'] = coverage.library + '::' + coverage.pathway
coverage = coverage[coverage.n_tested.between(10, min(300, len(universe_genes) - 1))]
gene_sets_all = {row.pathway_id: list(row.genes_present) for row in coverage.itertuples()}
if len(gene_sets_all) != 1513:
    raise ValueError(f'Expected notebook 37\'s 1,513 pathways, found {len(gene_sets_all)}.')
injury_ids = ['MSigDB_Hallmark_2020::' + name for name in INJURY_HALLMARKS]
print(f'{len(universe_genes):,} universe genes · {len(gene_sets_all):,} pathways')

## 3 · Unit-level inputs

Every dataset becomes the same form: raw counts (units × accepted orthologs, mouse symbols), donor,
segment and per-donor metadata.

- **Census mouse** (dataset 25818bf7, 10x multiome): nuclei annotated PT segment 1/2/3; embryonic and
  newborn stages removed, as in notebook 37. Donors are mice of four ages (3 weeks, 3–6 months,
  12 months, 21 months), 3 per age and sex.
- **Census human** (dataset 09b518f9, cortex): convoluted PT ("early") and S3 only.
- **Lake/KPMP** (CELLxGENE v1.5): healthy cortex nuclei labelled PT-S1/S2/S3, read from the raw
  counts block by block; Ensembl ids mapped to the ortholog map as in notebook 37.
- **Our cohort** (notebook 37's structures and normalization) and **our AKI mice** (mouse_only_v5
  PT structures with reviewed segment classes and the mouse-only PT DPT coordinate).

In [ ]:
from pseudospace.stage_cache import cached_anndata, cached_frame, cached_payload, digest
import pseudospace.specimen_null as specimen_null_module
import pseudospace.pathway_calibration as calibration_module

cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(specimen_null_module.__file__), Path(calibration_module.__file__)])
ortholog_mouse = set(orthologs.mouse_symbol)


def census_units(label, segment_map):
    data = ad.read_h5ad(census_cells[label])
    obs = data.obs.copy()
    keep = (~obs.development_stage.str.contains('embryo|E16|Theiler', case=False)) & obs.cell_type.isin(segment_map)
    data = data[keep.to_numpy()].copy()
    data.obs['segment'] = data.obs.cell_type.map(segment_map).astype(str)
    data.obs['donor'] = data.obs.donor_id.astype(str)
    return data


census_mouse = census_units('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                                      'epithelial cell of proximal tubule segment 2': 'S2',
                                      'epithelial cell of proximal tubule segment 3': 'S3'})
census_human = census_units('human', {'kidney proximal convoluted tubule epithelial cell': 'early',
                                      'epithelial cell of proximal tubule segment 3': 'S3'})


def lake_units():
    import h5py

    def column(group, name):
        item = group[name]
        if isinstance(item, h5py.Group):
            categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
            return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
        values = item[:]
        return values.astype(str) if values.dtype.kind in 'SO' else values

    with h5py.File(lake_file, 'r') as handle:
        obs = pd.DataFrame({c: column(handle['obs'], c) for c in ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(column(handle['raw/var'], 'feature_name'))
        keep = (obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3']) & obs.disease.eq('normal')
                & obs.region.eq('C') & obs.tissue.eq('cortex of kidney')).to_numpy()
        rows = np.flatnonzero(keep)
        raw = handle['raw/X']
        ptr = raw['indptr'][:].astype(np.int64)
        data_parts, index_parts, lengths = [], [], []
        block = 20000
        for first in range(0, len(ptr) - 1, block):
            last = min(first + block, len(ptr) - 1)
            chosen = rows[(rows >= first) & (rows < last)]
            if not len(chosen):
                continue
            values = raw['data'][ptr[first]:ptr[last]]
            columns = raw['indices'][ptr[first]:ptr[last]]
            for row in chosen:
                a, b = ptr[row] - ptr[first], ptr[row + 1] - ptr[first]
                data_parts.append(values[a:b]), index_parts.append(columns[a:b]), lengths.append(b - a)
        matrix = sparse.csr_matrix((np.concatenate(data_parts).astype(np.float32), np.concatenate(index_parts),
                                    np.concatenate([[0], np.cumsum(lengths)])), shape=(len(rows), len(var_ids)))
    human_source = ad.read_h5ad(data_root / 'tubule_by_gene' / 'HUK1_COR1_tubule_by_gene_caleb.h5ad', backed='r')
    human_var = human_source.var[['gene_ids']].copy()
    human_source.file.close()
    by_id = dict(zip(orthologs.human_symbol.map(human_var.gene_ids), orthologs.mouse_symbol))
    by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
    target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(var_ids, var_names)])
    mapped = np.flatnonzero(target != '')
    unique, codes = np.unique(target[mapped], return_inverse=True)
    collapse = sparse.csr_matrix((np.ones(len(mapped)), (mapped, codes)), shape=(len(var_ids), len(unique)))
    counts = (matrix @ collapse).tocsr().astype(np.float32)
    meta = obs.loc[keep, ['donor_id', 'sex', 'subclass.l2']].reset_index(drop=True)
    meta.index = meta.index.astype(str)
    meta['donor'] = meta.donor_id.astype(str)
    meta['segment'] = meta['subclass.l2'].str.replace('PT-', '', regex=False)
    return ad.AnnData(X=counts, obs=meta[['donor', 'sex', 'segment']], var=pd.DataFrame(index=unique))


lake = cached_anndata('lake_pt_cortex_units', lake_units, root=cache,
                      params={'labels': 'PT-S1,PT-S2,PT-S3', 'region': 'C', 'disease': 'normal'},
                      inputs={'lake': [lake_file.name, lake_file.stat().st_size], 'orthologs': digest(orthologs)}, code=code)


def complete_donors(obs, segments, minimum=MIN_UNITS):
    table = obs.groupby(['donor', 'segment'], observed=True).size().unstack(fill_value=0).reindex(columns=segments, fill_value=0)
    return sorted(table.index[(table >= minimum).all(axis=1)])


mouse_meta = (census_mouse.obs.drop_duplicates('donor').set_index('donor')[['sex', 'development_stage']]
              .rename(columns={'development_stage': 'age'}))
mouse_complete = complete_donors(census_mouse.obs, ['S1', 'S2', 'S3'])
mouse_male = [d for d in mouse_complete if mouse_meta.loc[d, 'sex'] == 'male']
mouse_female = [d for d in mouse_complete if mouse_meta.loc[d, 'sex'] == 'female']
lake_donors = complete_donors(lake.obs, ['S1', 'S2', 'S3'])
lake_meta = lake.obs.drop_duplicates('donor').set_index('donor')[['sex']]
human_census_donors = complete_donors(census_human.obs, ['early', 'S3'])
human_census_meta = census_human.obs.drop_duplicates('donor').set_index('donor')[['sex']]
print(f'Census mouse: {len(mouse_male)} male, {len(mouse_female)} female donors; Lake cortex: {len(lake_donors)} '
      f'donors {lake_meta.loc[lake_donors, "sex"].value_counts().to_dict()}; Census human cortex: {len(human_census_donors)} '
      f'donors {human_census_meta.loc[human_census_donors, "sex"].value_counts().to_dict()}')
display(mouse_meta.loc[mouse_complete].groupby(['sex', 'age']).size().unstack(fill_value=0))

### Our cohort (notebook 37's structures) and our AKI mice (mouse_only_v5)

Our cohort is rebuilt exactly as notebook 37 builds it (scFates coordinate support, original
specimen counts, ortholog library size). The AKI mice use mouse_only_v5's PT structures; that run
predates the fine re-segmentation, so its structure set differs from notebook 37's for the
control mice. Its counts layer is renormalized with the same rule.

In [ ]:
from pseudospace.pathway_inputs import rebuild_pt_expression

saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_obs = saved_pt.obs.copy()
saved_pt.file.close()
pt_obs = pt_obs[pt_obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
cohort_expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
support = pt_obs.groupby('sample', observed=True).shared_pseudospace.quantile([.01, .99]).unstack()
coordinate = pt_obs.shared_pseudospace.to_numpy(float)
pt_obs = pt_obs[(coordinate >= support[.01].max()) & (coordinate <= support[.99].min())].copy()
source_paths = {name: data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad'
                for names in cohort_expected.values() for name in names}
cohort = rebuild_pt_expression(pt_obs, source_paths, orthologs, target_sum=1e4)
cohort_units = ad.AnnData(X=sparse.csr_matrix(cohort.layers['counts'])[:, cohort.var.measured_in_both_inputs.to_numpy()],
                          obs=pd.DataFrame({'donor': cohort.obs['sample'].astype(str).to_numpy(),
                                            'segment': cohort.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy(),
                                            'position': cohort.obs.shared_pseudospace.to_numpy(float)},
                                           index=cohort.obs_names),
                          var=pd.DataFrame(index=cohort.var_names[cohort.var.measured_in_both_inputs.to_numpy()]))
cohort_meta = pd.DataFrame({'species': ['mouse', 'mouse', 'human', 'human']},
                           index=cohort_expected['mouse'] + cohort_expected['human'])

v5 = ad.read_h5ad(mouse_v5 / 'all_mouse_tubules_scanpy_dpt.h5ad')
v5 = v5[v5.obs.broad_tubule_marker_call.astype(str).eq('PT').to_numpy()
        & v5.obs.segment_class.astype(str).isin(['PT-S1', 'PT-S2', 'PT-S3']).to_numpy()].copy()
v5_support = v5.obs.groupby('sample', observed=True).pt_subset_scanpy_dpt.quantile([.01, .99]).unstack()
v5_coordinate = v5.obs.pt_subset_scanpy_dpt.to_numpy(float)
v5 = v5[(v5_coordinate >= v5_support[.01].max()) & (v5_coordinate <= v5_support[.99].min())].copy()
v5_position = v5.obs.pt_subset_scanpy_dpt.to_numpy(float)
v5_position = (v5_position - v5_position.min()) / (v5_position.max() - v5_position.min())
panel = np.isin(v5.var_names, list(ortholog_mouse))
aki_units = ad.AnnData(X=sparse.csr_matrix(v5.layers['counts'])[:, panel],
                       obs=pd.DataFrame({'donor': v5.obs['sample'].astype(str).to_numpy(),
                                         'segment': v5.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy(),
                                         'position': v5_position}, index=v5.obs_names),
                       var=pd.DataFrame(index=v5.var_names[panel]))
aki_meta = pd.DataFrame({'condition': ['control', 'control', 'AKI', 'AKI']}, index=['Ctrl1A2', 'Ctrl1A4', 'IR2A2', 'IR2A4'])
display(pd.concat({'our cohort': cohort_units.obs.groupby(['donor', 'segment']).size().unstack(),
                   'AKI design (v5)': aki_units.obs.groupby(['donor', 'segment']).size().unstack()}))

## 4 · Pools: normalization, tested genes, matching strata, pathway sets and VIF

A **pool** is the donor set from which draws are made. Everything that notebook 37 computes once
per cohort is computed once per pool and shared by every draw: tested genes (equal-donor
detection, species-balanced in the cross-species pool), the 3 × 3 × 3 matching strata (mean,
detection, coverage = share of segments with any detection), the pathway sets and the CAMERA
variance inflation (VIF). The VIF comes from unit-level residual correlations: units centred
within specimen × segment, mean member-pair correlation per specimen, Fisher-averaged, as in
notebook 12. Cell sizes, means, within-cell sums of squares and raw-count sums per
specimen × segment are kept; every draw is computed from them.

In [ ]:
from pseudospace.pathway_remodeling import (matching_strata, residual_pathway_correlations,
                                            correlation_adjusted_rank_tests)
from pseudospace.specimen_null import cell_summaries


def restrict_sets(tested):
    keep = set(tested)
    sets = {p: [g for g in v if g in keep] for p, v in gene_sets_all.items()}
    return {p: v for p, v in sets.items() if 10 <= len(v) <= min(300, len(keep) - 1)}


def build_pool(sources, species_of):
    """sources: list of (units AnnData, donors); species_of: donor -> species label."""
    panel = sorted(set.intersection(*[set(u.var_names[np.asarray(u.X.sum(axis=0)).ravel() > 0]) for u, _ in sources]))
    blocks, donor, segment = [], [], []
    for units, donors in sources:
        sub = units[units.obs.donor.isin(donors).to_numpy()]
        blocks.append(sparse.csr_matrix(sub[:, panel].X, dtype=float))
        donor.append(sub.obs.donor.astype(str).to_numpy()), segment.append(sub.obs.segment.astype(str).to_numpy())
    counts = sparse.vstack(blocks, format='csr')
    donor, segment = np.concatenate(donor), np.concatenate(segment)
    library = np.asarray(counts.sum(axis=1)).ravel()
    lognorm = counts.multiply((1e4 / library)[:, None]).tocsr()
    lognorm.data = np.log1p(lognorm.data)
    genes = [g for g in universe_genes if g in set(panel)]
    columns = pd.Index(panel).get_indexer(genes)
    x, c = lognorm[:, columns], counts[:, columns]
    donors = np.unique(donor)
    per_donor = {}
    for name in donors:
        rows = x[donor == name]
        seg = segment[donor == name]
        detected_by_segment = np.array([np.asarray((rows[seg == s] > 0).sum(axis=0)).ravel() > 0 for s in np.unique(seg)])
        per_donor[name] = (np.asarray(rows.mean(axis=0)).ravel(), np.asarray((rows > 0).mean(axis=0)).ravel(),
                           detected_by_segment.mean(axis=0))
    species = pd.Series({d: species_of[d] for d in donors})
    balanced = [np.mean([np.mean([per_donor[d][k] for d in species.index[species.eq(s)]], axis=0)
                         for s in species.unique()], axis=0) for k in range(3)]
    covariates = pd.DataFrame({'mean_expression': balanced[0], 'detection': balanced[1], 'coverage': balanced[2]}, index=genes)
    tested = covariates.detection.ge(DETECTION).to_numpy()
    tested_genes = list(np.array(genes)[tested])
    sets = restrict_sets(tested_genes)
    summary = cell_summaries(x[:, tested], donor, segment, c[:, tested], library)
    corr = []
    for name in donors:
        rows = x[donor == name][:, tested].toarray()
        seg = segment[donor == name]
        for s in np.unique(seg):
            rows[seg == s] -= rows[seg == s].mean(axis=0)
        corr.append(residual_pathway_correlations(rows, np.repeat(name, len(rows)), tested_genes, sets))
    corr = pd.concat(corr, ignore_index=True)
    reference = covariates.loc[tested_genes, 'mean_expression']
    camera = correlation_adjusted_rank_tests(reference, sets, corr).set_index('pathway_id')
    return {'donors': summary['donors'], 'segments': summary['segments'], 'n': summary['n'], 'mean': summary['mean'],
            'within': summary['within'], 'counts': summary['counts'], 'library': summary['library'],
            'genes': np.array(tested_genes), 'pathways': np.array(list(sets)),
            'strata': matching_strata(covariates.loc[tested_genes], bins=3),
            'vif': camera.variance_inflation.reindex(list(sets)).to_numpy(float),
            'rho_within': camera.rho_residual.reindex(list(sets)).to_numpy(float),
            'mean_expression': reference.to_numpy(float), 'n_units': np.array(len(donor))}


def pool(stage, sources, species_of):
    inputs = {'sources': [digest([digest(u.X), list(u.obs.donor), list(u.obs.segment), list(d)]) for u, d in sources],
              'universe_and_sets': digest([universe_genes, sorted(gene_sets_all.items())])}
    payload = cached_payload(stage, lambda: build_pool(sources, species_of), root=cache,
                             params={'detection': DETECTION, 'target_sum': 1e4}, inputs=inputs, code=code)
    restricted = restrict_sets(list(payload['genes']))
    if list(restricted) != list(payload['pathways']):
        raise ValueError(f'{stage}: cached pathways do not match the tested genes.')
    payload['sets'] = restricted
    return payload


pools = {
    'N1 Census mouse, male': pool('pool_census_mouse_male', [(census_mouse, mouse_male)], {d: 'mouse' for d in mouse_male}),
    'N2 Census mouse, female': pool('pool_census_mouse_female', [(census_mouse, mouse_female)], {d: 'mouse' for d in mouse_female}),
    'N3 Lake/KPMP human cortex': pool('pool_lake_human', [(lake, lake_donors)], {d: 'human' for d in lake_donors}),
    'N4 Census human cortex': pool('pool_census_human', [(census_human, human_census_donors)], {d: 'human' for d in human_census_donors}),
    'N5 Lake human + Census male mouse': pool('pool_cross_species', [(lake, lake_donors), (census_mouse, mouse_male)],
                                              {**{d: 'human' for d in lake_donors}, **{d: 'mouse' for d in mouse_male}}),
    'N7 our cohort': pool('pool_our_cohort', [(cohort_units, list(cohort_meta.index))], cohort_meta.species.to_dict()),
    'N6 our AKI design': pool('pool_aki', [(aki_units, list(aki_meta.index))], {d: 'mouse' for d in aki_meta.index}),
}
display(pd.DataFrame({name: {'donors': len(p['donors']), 'units': int(p['n_units']), 'tested genes': len(p['genes']),
                             'pathways': len(p['pathways']), 'median VIF': float(np.nanmedian(p['vif'])),
                             'median within-unit ρ': float(np.nanmedian(p['rho_within']))}
                      for name, p in pools.items()}).T)
if len(pools['N7 our cohort']['genes']) != len(universe_genes):
    raise ValueError('Our cohort should keep notebook 37\'s whole gene universe.')

## 5 · One partition, every strategy

For a partition (specimens, a 0/1 group per specimen, and for relabelings the true factor as a
0/1 nuisance) the notebook computes:

| Strategy | Family | Gene statistic | Pathway test |
|---|---|---|---|
| **O1** whole-PT pseudobulk + GSEA | average | moderated t, specimen sums | signed GSEA |
| **O2** S1/S2/S3 pseudobulk + GSEA | average | moderated t per segment | signed GSEA, any segment |
| **O3** `T_level` joint | average | unit-level offset partial F | matched rank-AUC ÷ √VIF |
| **P1** `T_steps` joint | position, steps | unit-level group × segment partial F | matched rank-AUC ÷ √VIF |
| `T_level` / `T_steps` matched | as above | as above | matched rank-AUC, no VIF |
| whole-PT / segment pseudobulk, joint rank test | average | moderated t | two-sided matched rank-AUC ÷ √VIF |

Partitions and GSEA runs are spread over worker processes (one BLAS and one gseapy thread each).

In [ ]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
import gseapy as gp
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
from pseudospace.specimen_null import matched_auc_null, pseudobulk_t, summary_partial_f

PRIMARY = ['O1 whole-PT pseudobulk + GSEA', 'O2 S1/S2/S3 pseudobulk + GSEA', 'O3 T_level joint', 'P1 T_steps joint']
SECONDARY = ['T_level matched', 'T_steps matched', 'whole-PT pseudobulk, joint rank test',
             'S1/S2/S3 pseudobulk, joint rank test']
STRATEGIES = PRIMARY + SECONDARY
_worker = {}


def _worker_init(pool_):
    from threadpoolctl import threadpool_limits
    threadpool_limits(1)
    _worker['pool'], _worker['sets'], _worker['genes'] = pool_, pool_['sets'], list(pool_['genes'])


def _partition_job(part):
    return partition_statistics(_worker['pool'], part['donors'], part['group'], part.get('nuisance'))


def _gsea_job(values):
    ranking = (pd.Series(values, index=_worker['genes']).rename_axis('gene').reset_index(name='stat')
               .sort_values(['stat', 'gene'], ascending=[False, True], kind='stable'))
    table = gp.prerank(rnk=ranking, gene_sets=_worker['sets'], min_size=10, max_size=300, weight=1,
                       method='multilevel', seed=GSEA_SEED, threads=1, outdir=None, no_plot=True, verbose=False).res2d
    return pd.Series(pd.to_numeric(table['NOM p-val']).to_numpy(float), index=table.Term.astype(str))


def partition_statistics(pool_, donors, group, nuisance=None):
    """Gene statistics and non-GSEA pathway p-values for one partition."""
    genes, sets, strata = pool_['genes'], pool_['sets'], pool_['strata']
    vif = pd.Series(pool_['vif'], index=pool_['pathways'])
    f = summary_partial_f(pool_, donors, group, nuisance)
    unit = matched_auc_null(pd.DataFrame({'T_level': f['T_level'], 'T_steps': f['T_steps']}, index=genes), sets, strata)
    index = pd.Index(pool_['donors']).get_indexer(donors)
    columns = [np.ones(len(donors)), [group[d] for d in donors]]
    if nuisance is not None:
        columns.append([nuisance[d] for d in donors])
    design = np.column_stack(columns).astype(float)
    t = {'whole': pseudobulk_t(pool_['counts'][index].sum(axis=1), pool_['library'][index].sum(axis=1), design, 1)[0]}
    for k, seg in enumerate(pool_['segments']):
        t[seg] = pseudobulk_t(pool_['counts'][index, k], pool_['library'][index, k], design, 1)[0]
    signed = matched_auc_null(pd.DataFrame(t, index=genes), sets, strata)
    p = {}
    for stat in ('T_level', 'T_steps'):
        rows = unit[unit.statistic.eq(stat)].set_index('pathway_id')
        positive = rows.effect.gt(0).to_numpy()
        p[f'{stat} matched'] = (np.asarray(norm.sf(rows.z_matched)), positive)
        p[f'{stat} joint'] = (np.asarray(norm.sf(rows.z_matched / np.sqrt(vif.loc[rows.index]))), positive)
    joint_signed = {k: 2 * norm.sf(np.abs(rows.z_matched.to_numpy() / np.sqrt(vif.loc[rows.pathway_id].to_numpy())))
                    for k, rows in signed.groupby('statistic', sort=False)}
    p['whole-PT pseudobulk, joint rank test'] = (joint_signed['whole'], None)
    p['S1/S2/S3 pseudobulk, joint rank test'] = (np.column_stack([joint_signed[s] for s in pool_['segments']]), None)
    return p, t


def bh_calls(p, positive=None):
    """BH within one strategy; a matrix is a pathway x segment family called if any segment passes."""
    p = np.asarray(p, float)
    q = multipletests(np.nan_to_num(p.ravel(), nan=1.), method='fdr_bh')[1].reshape(p.shape)
    hit = q <= ALPHA
    if hit.ndim == 2:
        hit = hit.any(axis=1)
    if positive is not None:
        hit &= positive
    return hit, float(np.mean(p <= .01))


def evaluate(pool_, partitions, stage, detail=False):
    """Calls of every strategy for every partition (cached). ``partitions``: list of dicts with
    label, donors, group, nuisance and any descriptive columns."""
    def compute():
        with ProcessPoolExecutor(max_workers=WORKERS, mp_context=multiprocessing.get_context('fork'),
                                 initializer=_worker_init, initargs=(pool_,)) as executor:
            prepared = list(executor.map(_partition_job, partitions, chunksize=2))
            jobs = [np.asarray(t[key], float) for _, t in prepared for key in ['whole', *pool_['segments']]]
            gsea = list(executor.map(_gsea_job, jobs, chunksize=4))
        pathways = list(pool_['pathways'])
        rows, details, cursor = [], [], 0
        for part, (p, _) in zip(partitions, prepared):
            k = len(pool_['segments'])
            g_whole = gsea[cursor].reindex(pathways).to_numpy(float)
            g_segments = np.column_stack([gsea[cursor + 1 + j].reindex(pathways).to_numpy(float) for j in range(k)])
            cursor += 1 + k
            p['O1 whole-PT pseudobulk + GSEA'] = (g_whole, None)
            p['O2 S1/S2/S3 pseudobulk + GSEA'] = (g_segments, None)
            p['O3 T_level joint'], p['P1 T_steps joint'] = p['T_level joint'], p['T_steps joint']
            for strategy in STRATEGIES:
                values, positive = p[strategy]
                hit, nominal = bh_calls(values, positive)
                rows.append({'label': part['label'], 'strategy': strategy, 'tested': len(pathways),
                             'calls': int(hit.sum()), 'nominal_p01': nominal,
                             'called': '|'.join(map(str, np.flatnonzero(hit))),
                             **{k_: v for k_, v in part.items() if k_ not in ('donors', 'group', 'nuisance', 'label')}})
                if detail:
                    best = values.min(axis=1) if np.ndim(values) == 2 else values
                    details.append(pd.DataFrame({'label': part['label'], 'strategy': strategy, 'pathway_id': pathways,
                                                 'p': best, 'hit': hit}))
        frame = pd.DataFrame(rows).assign(kind='summary')
        if detail:
            frame = pd.concat([frame, pd.concat(details, ignore_index=True).assign(kind='detail')], ignore_index=True)
        return frame
    spec = [{k: (sorted(v.items()) if isinstance(v, dict) else v) for k, v in part.items()} for part in partitions]
    result = cached_frame(stage, compute, root=cache, params={'alpha': ALPHA, 'gsea_seed': GSEA_SEED},
                          inputs={'pool': [digest(pool_['mean']), list(pool_['pathways'])], 'partitions': spec}, code=code)
    detail_only = ['pathway_id', 'p', 'hit']
    summary = result[result.kind.eq('summary')].drop(columns=['kind', *[c for c in detail_only if c in result]])
    summary = summary.assign(called=summary.called.fillna('').astype(str)).reset_index(drop=True)
    if not detail:
        return summary
    return summary, result.loc[result.kind.eq('detail'), ['label', 'strategy', *detail_only]].reset_index(drop=True)


def balanced_splits(four):
    a, b, c, d = four
    return [((a, b), (c, d)), ((a, c), (b, d)), ((a, d), (b, c))]


def same_species_partitions(donors, n_sets, attribute, rng):
    all_sets = list(combinations(sorted(donors), 4))
    chosen = all_sets if len(all_sets) <= n_sets else [all_sets[i] for i in sorted(rng.choice(len(all_sets), n_sets, replace=False))]
    parts = []
    for i, four in enumerate(chosen):
        for j, (first, second) in enumerate(balanced_splits(four)):
            balanced = sorted(attribute[list(first)]) == sorted(attribute[list(second)])
            parts.append({'label': f'set{i:03d}_split{j}', 'donors': list(four),
                          'group': {d: float(d in second) for d in four}, 'balanced': bool(balanced),
                          'set': i})
    return parts

## 6 · D5 · Same-species null draws

Four donors of one species split into two pairs have no group difference in expectation: every
call is a specimen call. Each 4-donor set gives its three balanced splits. Splits are labelled
**balanced** when both pairs have the same age composition (mouse) or sex composition (human).

In [ ]:
rng = np.random.default_rng(SEED)
null_specs = {
    'N1 Census mouse, male': (mouse_male, mouse_meta.age),
    'N2 Census mouse, female': (mouse_female, mouse_meta.age),
    'N3 Lake/KPMP human cortex': (lake_donors, lake_meta.sex),
    'N4 Census human cortex': (human_census_donors, human_census_meta.sex),
}
null_results, null_parts = {}, {}
for name, (donors, attribute) in null_specs.items():
    parts = null_parts[name] = same_species_partitions(donors, N_SETS, attribute, rng)
    stage = 'null_' + name.split()[0]
    null_results[name] = evaluate(pools[name], parts, stage).assign(pool=name)
null_calls = pd.concat(null_results.values(), ignore_index=True)
null_calls.drop(columns='called').to_csv(tables / 'same_species_null_calls.csv', index=False)


def distribution(frame):
    return pd.Series({'splits': len(frame), 'tested': int(frame.tested.iloc[0]),
                      'median': frame.calls.median(), 'mean': frame.calls.mean(),
                      'p90': frame.calls.quantile(.9), 'max': frame.calls.max(),
                      'share ≥ 1 call': frame.calls.ge(1).mean(),
                      'share ≥ 5% of tested': (frame.calls / frame.tested).ge(NULL_RATE).mean(),
                      'mean share p ≤ 0.01': frame.nominal_p01.mean()})


null_summary = (null_calls.groupby(['pool', 'strategy'], sort=False).apply(distribution).reset_index())
null_summary.to_csv(tables / 'table_null_call_distribution.csv', index=False)
display(null_summary[null_summary.strategy.isin(PRIMARY)].round(3))
display(null_summary[null_summary.strategy.isin(SECONDARY)].round(3))
balance_summary = (null_calls[null_calls.strategy.isin(PRIMARY)]
                   .groupby(['pool', 'strategy', 'balanced'], sort=False).calls
                   .agg(['size', 'median', 'mean', lambda c: c.quantile(.9)]).rename(columns={'<lambda_0>': 'p90'}))
balance_summary.to_csv(tables / 'null_calls_by_composition_balance.csv')
display(balance_summary.round(2))

### Post hoc (not pre-specified): prepubertal mice

The mouse atlas's authors report that sex-biased PT genes emerge after 3 weeks of age, with
distinct patterns in the outer stripe (https://pubmed.ncbi.nlm.nih.gov/40259083/). A split that puts
more 3-week-old mice on one side than the other then contrasts pre- and post-pubertal PT, a
genuine specimen difference that is partly S3-specific. This check, added after the pooled
results were seen, splits the mouse null draws by that imbalance (0, 1 or 2 more juveniles on
one side).

In [ ]:
juvenile = set(mouse_meta.index[mouse_meta.age.str.contains('21-day')])
puberty_rows = []
for name in ['N1 Census mouse, male', 'N2 Census mouse, female']:
    imbalance = {part['label']: abs(sum((d in juvenile) * (2 * part['group'][d] - 1) for d in part['donors']))
                 for part in null_parts[name]}
    frame = null_calls[null_calls.pool.eq(name) & null_calls.strategy.isin(PRIMARY)]
    frame = frame.assign(juvenile_imbalance=frame.label.map(imbalance))
    puberty_rows.append(frame.groupby(['strategy', 'juvenile_imbalance']).calls
                        .agg(['size', 'median', 'mean', lambda c: c.quantile(.9)]).rename(columns={'<lambda_0>': 'p90'})
                        .assign(pool=name).reset_index())
puberty = pd.concat(puberty_rows, ignore_index=True)
puberty.to_csv(tables / 'posthoc_null_calls_by_juvenile_imbalance.csv', index=False)
display(puberty.pivot_table(index=['pool', 'strategy'], columns='juvenile_imbalance', values='median'))

## 7 · D5 · Cross-species 2 + 2 draws and an all-donor reference

Each draw takes 2 of the 7 Lake human donors and 2 of the 12 Census male mice, normalized on the
orthologs measured in both datasets. As in our cohort, the species split and its two balanced
relabelings are run; in the relabelings the species' segment shape is a nuisance term. The
all-donor analysis (7 human against 12 mouse donors, same machinery) is the reference for
precision: of a draw's species calls, the share also called by the reference, compared with the
share of all pathways the reference calls. The reference is not ground truth: it uses the same
tests, only with more donors.

In [ ]:
cross = pools['N5 Lake human + Census male mouse']
species_of = {**{d: 1. for d in lake_donors}, **{d: 0. for d in mouse_male}}
pairs_h, pairs_m = list(combinations(lake_donors, 2)), list(combinations(mouse_male, 2))
all_draws = [(h, m) for h in pairs_h for m in pairs_m]
rng_cross = np.random.default_rng(SEED + 1)
draws = [all_draws[i] for i in sorted(rng_cross.choice(len(all_draws), N_CROSS, replace=False))]
cross_parts = []
for i, ((h1, h2), (m1, m2)) in enumerate(draws):
    four = [h1, h2, m1, m2]
    cross_parts.append({'label': f'draw{i:03d}_species', 'donors': four, 'group': {d: species_of[d] for d in four},
                        'draw': i, 'partition': 'species'})
    for j, partner in enumerate((m1, m2)):
        first = {h1, partner}
        cross_parts.append({'label': f'draw{i:03d}_relabel{j}', 'donors': four,
                            'group': {d: float(d in first) for d in four}, 'nuisance': {d: species_of[d] for d in four},
                            'draw': i, 'partition': f'relabel{j}'})
cross_calls = evaluate(cross, cross_parts, 'cross_species_draws')
reference_calls = evaluate(cross, [{'label': 'all_donors', 'donors': list(lake_donors) + list(mouse_male),
                                    'group': species_of, 'partition': 'reference'}], 'cross_species_reference')
reference_sets = {row.strategy: set(filter(None, row.called.split('|'))) for row in reference_calls.itertuples()}


def draw_metrics(frame):
    species = frame[frame.partition.eq('species')].iloc[0]
    relabel = frame[frame.partition.ne('species')]
    called = set(filter(None, species.called.split('|')))
    ref = reference_sets[species.strategy]
    return pd.Series({'species_calls': species.calls, 'relabel_mean': relabel.calls.mean(),
                      'relabel_max': relabel.calls.max(),
                      'ncdr': relabel.calls.mean() / species.calls if species.calls else np.nan,
                      'meets_nb31_rule': bool(species.calls and relabel.calls.mean() / species.calls < .25
                                              and relabel.calls.mean() / species.tested < NULL_RATE),
                      'precision_vs_reference': len(called & ref) / len(called) if called else np.nan,
                      'reference_rate': len(ref) / species.tested,
                      'relabel_nominal_p01': relabel.nominal_p01.mean()})


cross_draws = cross_calls.groupby(['strategy', 'draw'], sort=False).apply(draw_metrics).reset_index()
cross_draws.to_csv(tables / 'cross_species_draws.csv', index=False)
cross_summary = cross_draws.groupby('strategy', sort=False).agg(
    draws=('draw', 'size'), species_calls_median=('species_calls', 'median'),
    relabel_calls_median=('relabel_mean', 'median'), relabel_calls_p90=('relabel_mean', lambda v: v.quantile(.9)),
    draws_with_species_calls=('species_calls', lambda v: int((v > 0).sum())),
    ncdr_median=('ncdr', 'median'), ncdr_p90=('ncdr', lambda v: v.quantile(.9)),
    share_meeting_nb31_rule=('meets_nb31_rule', 'mean'), precision_median=('precision_vs_reference', 'median'),
    reference_rate=('reference_rate', 'first'), relabel_nominal_p01=('relabel_nominal_p01', 'mean'))
cross_summary['reference_calls'] = [len(reference_sets[s]) for s in cross_summary.index]
cross_summary.to_csv(tables / 'table_cross_species_draws_summary.csv')
display(cross_summary.round(3))

## 8 · Anchor: our cohort with the same machinery, and a check of the exact matched null

Our cohort's species split and its two relabelings (one human section with each mouse) are run
through exactly the code used for the public draws, so our two relabelings can be placed in the
external null distributions. Notebook 37's numbers (DESeq2, spline-based designs, 9,999 random
sets) are shown beside them.

The exact matched-null moments replace notebook 37's 9,999 random sets. As a check on real data,
notebook 37's species `T_spatial` pathway z is recomputed with its own strata and compared.

In [ ]:
from pseudospace.pathway_remodeling import gene_covariates, matching_strata as _strata

ours = pools['N7 our cohort']
human_specimens, mouse_specimens = cohort_expected['human'], cohort_expected['mouse']
is_human = {d: float(d in human_specimens) for d in cohort_meta.index}
anchor_parts = [{'label': 'species', 'donors': list(cohort_meta.index), 'group': is_human, 'partition': 'species'}]
for partner in mouse_specimens:
    first = {human_specimens[0], partner}
    anchor_parts.append({'label': 'relabel_' + partner, 'donors': list(cohort_meta.index),
                         'group': {d: float(d in first) for d in cohort_meta.index}, 'nuisance': is_human,
                         'partition': 'relabel'})
anchor_calls = evaluate(ours, anchor_parts, 'anchor_our_cohort')
anchor_table = anchor_calls.pivot(index='strategy', columns='label', values='calls').reindex(STRATEGIES)
nb37_table = pd.read_csv(upstream37 / 'table_S1_strategy_specificity.csv', index_col=0)
nb37_rows = {'O1 whole-PT pseudobulk + GSEA': '1 · Whole-PT pseudobulk + signed GSEA',
             'O2 S1/S2/S3 pseudobulk + GSEA': '2 · S1/S2/S3 pseudobulk + signed GSEA',
             'O3 T_level joint': '4 · Gene GAM offset (T_level) (joint test)',
             'P1 T_steps joint': '6 · Gene GAM × S1/S2/S3 steps (joint test)',
             'T_level matched': '4 · Gene GAM offset (T_level)', 'T_steps matched': '6 · Gene GAM × S1/S2/S3 steps'}
for strategy, row in nb37_rows.items():
    anchor_table.loc[strategy, 'nb37 species'] = nb37_table.loc[row, 'species_calls']
    anchor_table.loc[strategy, 'nb37 relabels'] = (f"{int(nb37_table.loc[row, 'relabel_calls_Ctrl1A2'])}, "
                                                   f"{int(nb37_table.loc[row, 'relabel_calls_Ctrl1A4'])}")
cross_relabels = cross_calls[cross_calls.partition.ne('species')]
for label in [c for c in anchor_table.columns if str(c).startswith('relabel_')]:
    anchor_table['percentile in N5 relabelings, ' + label] = [
        float((cross_relabels.loc[cross_relabels.strategy.eq(s), 'calls'] <= anchor_table.loc[s, label]).mean())
        for s in anchor_table.index]
anchor_table.to_csv(tables / 'anchor_our_cohort.csv')
display(anchor_table)

nb37_tests = pd.read_csv(upstream37 / 'matched_and_joint_tests_all_partitions.csv')
nb37_spatial = nb37_tests[nb37_tests.partition.eq('species') & nb37_tests.statistic.eq('T_spatial')].set_index('pathway_id')
covariates37 = gene_covariates(sparse.csr_matrix(cohort.layers['lognorm']), cohort.obs.shared_pseudospace.to_numpy(float),
                               cohort.obs.comparison_species.astype(str).eq('human').to_numpy(),
                               cohort.obs['sample'].astype(str).to_numpy(), cohort.var_names).set_index('gene').loc[universe_genes]
t_spatial37 = pd.read_csv(upstream37 / 'gene_level_statistics.csv', index_col=0).T_spatial.reindex(universe_genes)
exact37 = matched_auc_null(t_spatial37.to_frame('T_spatial'), gene_sets_all, _strata(covariates37, bins=3)).set_index('pathway_id')
exact37 = exact37.loc[nb37_spatial.index]
exact_joint = norm.sf(exact37.z_matched / np.sqrt(nb37_spatial.vif))
exact_calls = int((exact37.effect.gt(0) & (multipletests(exact_joint, method='fdr_bh')[1] <= ALPHA)).sum())
check = pd.Series({'max |Δ AUC|': float((exact37.auc - nb37_spatial.auc).abs().max()),
                   'median |Δ z| (exact − 9,999 sets)': float((exact37.z_matched - nb37_spatial.z_matched).abs().median()),
                   'max |Δ z|': float((exact37.z_matched - nb37_spatial.z_matched).abs().max()),
                   'joint calls, exact null': exact_calls,
                   'joint calls, notebook 37': int((nb37_spatial.effect.gt(0) & nb37_spatial.q_joint.le(ALPHA)).sum())})
check.to_csv(tables / 'exact_matched_null_check.csv')
display(check.to_frame('value'))

## 9 · Mechanism: how coherent are specimen deviations within pathways?

The split-plot df count (2 for offsets, 2(k − 1) for positional terms) concerns specimen-level
tests. The screens here are competitive tests on unit-level statistics, and they fail when a
specimen difference moves a pathway's genes **together**, a correlation that unit-level residuals
(the VIF) cannot see. From all donors of each pool, each gene's cell means are split into a
specimen offset and a specimen × segment deviation (two-way decomposition, equal segment
weights). For every pathway, the mean pairwise correlation of its genes across specimens
(offsets) or across specimen × segment cells (deviations) is compared with the within-cell
residual correlation that enters the VIF. The per-gene ratio of offset to deviation variance is
also reported. In our cohort and the AKI design, each species' or condition's mean profile is
removed first, so only deviations between specimens of the same group are measured.

In [ ]:
from scipy.stats import spearmanr
from pseudospace.specimen_null import donor_deviations

mechanism_rows, coherence_frames = [], []
within_group = {'N7 our cohort': cohort_meta.species.to_dict(), 'N6 our AKI design': aki_meta.condition.to_dict()}
for name in [*null_specs, 'N7 our cohort', 'N6 our AKI design']:
    p_ = pools[name]
    offsets, deviations = donor_deviations(p_, list(p_['donors']), within_group.get(name))
    genes_ = list(p_['genes'])
    coherence = {}
    for label, matrix in (('offset', offsets), ('specimen × segment', deviations)):
        corr = residual_pathway_correlations(matrix, np.repeat('all', len(matrix)), genes_, p_['sets'])
        coherence[label] = corr.set_index('pathway_id').rho_specimen.reindex(list(p_['pathways']))
    coherence['within-unit (VIF)'] = pd.Series(p_['rho_within'], index=p_['pathways'])
    frame = pd.DataFrame(coherence).assign(pool=name)
    coherence_frames.append(frame.rename_axis('pathway_id').reset_index())
    k = len(p_['segments'])
    variance_offset = offsets.var(axis=0, ddof=1)
    variance_deviation = (deviations ** 2).sum(axis=0) / max((len(p_['donors']) - 1) * (k - 1), 1)
    mechanism_rows.append({'pool': name, 'specimens': len(p_['donors']),
                           **{f'median ρ, {c}': float(frame[c].median()) for c in coherence},
                           'median per-gene offset ÷ deviation variance': float(np.median(variance_offset / np.maximum(variance_deviation, 1e-12)))})
mechanism = pd.DataFrame(mechanism_rows).set_index('pool')
coherence_all = pd.concat(coherence_frames, ignore_index=True)
coherence_all.to_csv(tables / 'pathway_coherence_by_pool.csv', index=False)
mechanism.to_csv(tables / 'table_mechanism_coherence.csv')
display(mechanism.round(3))

frequency_rows = []
for name in null_specs:
    p_ = pools[name]
    rows_ = null_calls[null_calls.pool.eq(name)]
    for strategy in ('O3 T_level joint', 'P1 T_steps joint', 'O1 whole-PT pseudobulk + GSEA'):
        counts_ = np.zeros(len(p_['pathways']))
        for called in rows_.loc[rows_.strategy.eq(strategy), 'called']:
            for i in filter(None, called.split('|')):
                counts_[int(i)] += 1
        frame = coherence_all[coherence_all.pool.eq(name)].set_index('pathway_id').reindex(list(p_['pathways']))
        column = 'specimen × segment' if strategy.startswith('P1') else 'offset'
        frequency_rows.append({'pool': name, 'strategy': strategy, 'coherence used': column,
                               'Spearman(null-call frequency, coherence)': spearmanr(counts_, frame[column], nan_policy='omit').statistic})
frequency = pd.DataFrame(frequency_rows)
frequency.to_csv(tables / 'null_frequency_vs_coherence.csv', index=False)
display(frequency.round(3))

## 10 · D6 · Positive control: AKI against control mice (2 + 2)

Ischaemia–reperfusion injures the S3 segment in the outer stripe most. A position-dependent screen
should therefore call injury programs for AKI against control, and they should sit in S3. The same
design gives two more relabelings, each pairing one control with one AKI mouse, with biological
variance on both sides.

Strategies: the eight above at segment resolution, plus notebook 37's coordinate `T_spatial`
(6-df spline on the mouse-only PT DPT, segment classes as steps in the designs, joint test with the
pool VIF).

In [ ]:
from pseudospace.pathway_calibration import contrast_designs, nested_partial_f
from pseudospace.stats_gam import gam_internal_knots

aki = pools['N6 our AKI design']
is_aki = {d: float(aki_meta.loc[d, 'condition'] == 'AKI') for d in aki_meta.index}
aki_parts = [{'label': 'condition', 'donors': list(aki_meta.index), 'group': is_aki, 'partition': 'condition'}]
for partner in ('IR2A2', 'IR2A4'):
    first = {'Ctrl1A2', partner}
    aki_parts.append({'label': 'relabel_Ctrl1A2+' + partner, 'donors': list(aki_meta.index),
                      'group': {d: float(d in first) for d in aki_meta.index}, 'nuisance': is_aki, 'partition': 'relabel'})
aki_calls, aki_detail = evaluate(aki, aki_parts, 'aki_partitions', detail=True)

aki_counts = sparse.csr_matrix(aki_units.X, dtype=float)
aki_library = np.asarray(aki_counts.sum(axis=1)).ravel()
aki_lognorm = aki_counts.multiply((1e4 / aki_library)[:, None]).tocsr()
aki_lognorm.data = np.log1p(aki_lognorm.data)
aki_y = aki_lognorm[:, aki_units.var_names.get_indexer(aki['genes'])]
aki_specimen = aki_units.obs.donor.astype(str).to_numpy()
aki_segment = aki_units.obs.segment.astype(str).to_numpy()
aki_position = aki_units.obs.position.to_numpy(float)
aki_knots = gam_internal_knots(aki_position, basis_df=6)
aki_vif = pd.Series(aki['vif'], index=aki['pathways'])


def spatial_partition(part):
    group = np.array([part['group'][d] for d in aki_specimen])
    nuisance = np.array([part['nuisance'][d] for d in aki_specimen]) if 'nuisance' in part else None
    designs, weights = contrast_designs(aki_position, group, aki_specimen, aki_knots, segments=aki_segment,
                                        nuisance_shape=nuisance)
    return {'T_spatial': nested_partial_f(aki_y, designs, weights, {'T_spatial': ('level', 'full')})['T_spatial']}


spatial_rows = []
for part in aki_parts:
    scores = cached_payload('aki_spatial_' + part['label'].replace('+', '_'), lambda part=part: spatial_partition(part),
                            root=cache, params={'basis_df': 6},
                            inputs={'y': digest(aki_y), 'position': aki_position, 'part': sorted(part['group'].items())}, code=code)
    rows = matched_auc_null(pd.DataFrame({'T_spatial': scores['T_spatial']}, index=aki['genes']), aki['sets'], aki['strata'])
    rows = rows.set_index('pathway_id')
    p_joint = norm.sf(rows.z_matched / np.sqrt(aki_vif.loc[rows.index]))
    hit = rows.effect.gt(0).to_numpy() & (multipletests(p_joint, method='fdr_bh')[1] <= ALPHA)
    spatial_rows.append(pd.DataFrame({'label': part['label'], 'strategy': 'T_spatial joint (coordinate)',
                                      'pathway_id': rows.index, 'p': p_joint, 'hit': hit}))
aki_detail = pd.concat([aki_detail, *spatial_rows], ignore_index=True)
aki_detail['hit'] = aki_detail.hit.astype(str).eq('True')
aki_table = aki_detail.groupby(['strategy', 'label']).hit.sum().unstack().reindex([*STRATEGIES, 'T_spatial joint (coordinate)'])
aki_table['relabel ÷ condition'] = aki_table[[c for c in aki_table if c.startswith('relabel')]].mean(axis=1) / aki_table.condition.where(aki_table.condition > 0)
aki_table.to_csv(tables / 'table_aki_strategy_calls.csv')
display(aki_table)

### Is the AKI effect where anatomy puts it?

**Segments.** For each tested injury marker, the AKI − control difference of equal-specimen
segment means. **Outer stripe proxy.** A structure's depth is its mean distance to the three
nearest glomeruli of the same section (mouse_only_v5 pass-1 centroids); S3 deeper than the
section's S1 95th percentile is "deep S3", the rest "cortical S3" (notebook 37's rule).
**Pathways.** The injury Hallmarks' rank among all pathways in each screen, and, for every pathway
a positional screen calls, the segment where its members' median AKI − control difference is
largest.

In [ ]:
from scipy.spatial import cKDTree
from scipy.stats import mannwhitneyu

markers = [g for g in INJURY_MARKERS if g in set(aki['genes'])]
marker_index = pd.Index(aki['genes']).get_indexer(markers)
controls, injured = ['Ctrl1A2', 'Ctrl1A4'], ['IR2A2', 'IR2A4']
donor_index = pd.Index(aki['donors'])
segment_mean = lambda donors: aki['mean'][donor_index.get_indexer(donors)].mean(axis=0)   # segments x genes
segment_delta = pd.DataFrame((segment_mean(injured) - segment_mean(controls))[:, marker_index].T,
                             index=markers, columns=aki['segments'])
segment_delta['largest increase'] = segment_delta[list(aki['segments'])].idxmax(axis=1)

pass1 = ad.read_h5ad(mouse_v5 / 'all_mouse_tubules_harmony_pass1.h5ad', backed='r')
anchors = pass1.obs[['sample', 'coarse_class', 'x_centroid', 'y_centroid']].copy()
pass1.file.close()
missing = aki_units.obs_names.difference(anchors.index)
if len(missing):
    raise ValueError(f'{len(missing)} AKI-design structures have no pass-1 centroid.')
depth = np.full(len(aki_specimen), np.nan)
for name in aki_meta.index:
    mask = aki_specimen == name
    glomeruli = anchors[anchors['sample'].astype(str).eq(name) & anchors.coarse_class.astype(str).eq('Glomerulus')]
    distance, _ = cKDTree(glomeruli[['x_centroid', 'y_centroid']].to_numpy(float)).query(
        anchors.loc[aki_units.obs_names[mask], ['x_centroid', 'y_centroid']].to_numpy(float), k=3)
    depth[mask] = distance.mean(axis=1)
region = aki_segment.astype(object).copy()
for name in aki_meta.index:
    limit = np.quantile(depth[(aki_specimen == name) & (aki_segment == 'S1')], .95)
    deep = (aki_specimen == name) & (aki_segment == 'S3') & (depth > limit)
    region[deep] = 'S3 deep'
    region[(aki_specimen == name) & (aki_segment == 'S3') & ~deep] = 'S3 cortical'
regions = ['S1', 'S2', 'S3 cortical', 'S3 deep']
marker_values = aki_y[:, marker_index].toarray()
region_means = {(d, r): marker_values[(aki_specimen == d) & (region == r)].mean(axis=0) for d in aki_meta.index for r in regions}
region_delta = pd.DataFrame({r: np.mean([region_means[(d, r)] for d in injured], axis=0)
                             - np.mean([region_means[(d, r)] for d in controls], axis=0) for r in regions}, index=markers)
region_counts = pd.crosstab(aki_specimen, region).reindex(columns=regions)
marker_check = pd.Series({'injury markers tested': len(markers),
                          'largest increase in S3': int(segment_delta['largest increase'].eq('S3').sum()),
                          'share in S3': float(segment_delta['largest increase'].eq('S3').mean()),
                          'median Δ S1': float(segment_delta['S1'].median()), 'median Δ S3': float(segment_delta['S3'].median()),
                          'median Δ deep S3': float(region_delta['S3 deep'].median()),
                          'median Δ cortical S3': float(region_delta['S3 cortical'].median()),
                          'markers with deep S3 > cortical S3': int((region_delta['S3 deep'] > region_delta['S3 cortical']).sum())})
segment_delta.join(region_delta.add_prefix('region ')).to_csv(tables / 'aki_injury_markers_by_segment_and_depth.csv')
region_counts.to_csv(tables / 'aki_structures_by_region.csv')
display(region_counts)
display(segment_delta.join(region_delta.add_prefix('region ')).round(3))
display(marker_check.to_frame('value'))

condition_detail = aki_detail[aki_detail.label.eq('condition')]
pathway_rows = []
member_delta = pd.DataFrame((segment_mean(injured) - segment_mean(controls)).T, index=aki['genes'], columns=aki['segments'])
for strategy, rows in condition_detail.groupby('strategy', sort=False):
    rows = rows.set_index('pathway_id')
    injured_ids = [p for p in injury_ids if p in rows.index]
    called = rows.index[rows.hit]
    located = pd.Series({p: member_delta.loc[aki['sets'][p]].median().abs().idxmax() for p in called}, dtype=object)
    pathway_rows.append({'strategy': strategy, 'condition calls': len(called),
                         'injury Hallmarks tested': len(injured_ids),
                         'injury Hallmarks called': int(rows.loc[injured_ids, 'hit'].sum()),
                         'called injury Hallmarks': '; '.join(p.split('::')[1] for p in injured_ids if rows.loc[p, 'hit']),
                         'Mann–Whitney p (injury Hallmarks rank higher)': mannwhitneyu(
                             rows.loc[injured_ids, 'p'], rows.drop(index=injured_ids).p, alternative='less').pvalue,
                         'called pathways with largest change in S3': float(located.eq('S3').mean()) if len(located) else np.nan})
aki_pathways = pd.DataFrame(pathway_rows).set_index('strategy').reindex([*STRATEGIES, 'T_spatial joint (coordinate)'])
aki_pathways.to_csv(tables / 'aki_injury_pathways.csv')
display(aki_pathways)

## 11 · Pre-specified decisions

In [ ]:
same_pools = list(null_specs)
p1 = null_summary[null_summary.strategy.eq('P1 T_steps joint')].set_index('pool')
p1_ok = (p1.p90 / p1.tested).le(NULL_RATE)
cross_p1, cross_o1 = cross_summary.loc['P1 T_steps joint'], cross_summary.loc['O1 whole-PT pseudobulk + GSEA']
general = (cross_p1.ncdr_median < .25 and cross_p1.share_meeting_nb31_rule >= .8 and cross_o1.ncdr_median >= .5
           and int(p1_ok.sum()) >= 3)
cohort_specific = cross_p1.ncdr_median >= .5 or int((~p1_ok).sum()) >= 2
R2_DECISION = 'general at segment resolution' if general else ('cohort-specific' if cohort_specific else 'partly general')
condition_row = aki_table.loc[['P1 T_steps joint', 'T_spatial joint (coordinate)']]
positional_pass = []
for strategy in ('P1 T_steps joint', 'T_spatial joint (coordinate)'):
    calls_ = aki_table.loc[strategy, 'condition']
    relabel_ = aki_table.loc[strategy, [c for c in aki_table if c.startswith('relabel_')]].mean()
    hallmarks_ok = (aki_pathways.loc[strategy, 'injury Hallmarks called'] >= 3
                    or aki_pathways.loc[strategy, 'Mann–Whitney p (injury Hallmarks rank higher)'] <= .01)
    positional_pass.append(bool(hallmarks_ok and calls_ > 0 and relabel_ <= .05 * calls_))
D6_PASS = any(positional_pass) and marker_check['share in S3'] >= 2 / 3
decisions = pd.Series({
    'P1 90th-percentile null calls ≤ 5% of tested (same-species pools)': f'{int(p1_ok.sum())} of {len(p1_ok)}',
    'N5 median NCDR, P1 T_steps joint': cross_p1.ncdr_median,
    'N5 share of draws meeting notebook 31 rule, P1': cross_p1.share_meeting_nb31_rule,
    'N5 median NCDR, O1 whole-PT pseudobulk + GSEA': cross_o1.ncdr_median,
    'R2 decision': R2_DECISION,
    'D6 positional screen passes (P1, T_spatial)': str(positional_pass),
    'D6 injury markers rising most in S3': f"{marker_check['largest increase in S3']} of {marker_check['injury markers tested']}",
    'D6 decision': 'pass' if D6_PASS else 'fail'})
decisions.to_csv(tables / 'decisions.csv')
display(decisions.to_frame('value'))

## 12 · Figure R2-revision · Specificity over many draws, its mechanism and the AKI positive control

(a) Same-species null calls per split, by pool and primary strategy (symmetric-log axis; box =
quartiles, whiskers = 5th–95th percentiles; the dashed line marks 5% of tested pathways).
(b) Cross-species 2 + 2 draws: species calls against mean relabeled calls per draw.
(c) Coherence of specimen deviations within pathways. (d) AKI against control: calls for the
condition and the two relabelings. (e) AKI − control difference of injury markers by region.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

AVERAGE, POSITION, MUTED, INK = '#eb6834', '#2a78d6', '#8a8984', '#0b0b0b'
plt.rcParams.update({'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK, 'xtick.color': INK, 'ytick.color': INK})
family_color = {s: (POSITION if s.startswith('P1') or s.startswith('T_steps') or 'T_spatial' in s else AVERAGE)
                for s in [*STRATEGIES, 'T_spatial joint (coordinate)']}
short = {'O1 whole-PT pseudobulk + GSEA': 'O1 whole-PT PB + GSEA', 'O2 S1/S2/S3 pseudobulk + GSEA': 'O2 segment PB + GSEA',
         'O3 T_level joint': 'O3 offset, joint', 'P1 T_steps joint': 'P1 steps, joint',
         'T_level matched': 'offset, matched', 'T_steps matched': 'steps, matched',
         'whole-PT pseudobulk, joint rank test': 'whole-PT PB, joint', 'S1/S2/S3 pseudobulk, joint rank test': 'segment PB, joint',
         'T_spatial joint (coordinate)': 'T_spatial, joint'}

fig = plt.figure(figsize=(7.2, 8.4))
grid_ = fig.add_gridspec(3, 2, height_ratios=[1.25, 1, 1], hspace=.75, wspace=.45)

ax = fig.add_subplot(grid_[0, :])
positions, labels_ = [], []
for i, name in enumerate(same_pools):
    for j, strategy in enumerate(PRIMARY):
        values = null_calls.loc[null_calls.pool.eq(name) & null_calls.strategy.eq(strategy), 'calls'].to_numpy()
        y = i * (len(PRIMARY) + 1.2) + j
        q5, q25, q50, q75, q95 = np.percentile(values, [5, 25, 50, 75, 95])
        ax.plot([q5, q95], [y, y], color=family_color[strategy], lw=1)
        ax.add_patch(plt.Rectangle((q25, y - .3), max(q75 - q25, 1e-3), .6, facecolor=family_color[strategy],
                                   alpha=.35, edgecolor=family_color[strategy], lw=1))
        ax.plot([q50, q50], [y - .3, y + .3], color=family_color[strategy], lw=2)
        positions.append(y), labels_.append(short[strategy])
    tested_ = int(null_calls.loc[null_calls.pool.eq(name), 'tested'].iloc[0])
    top = i * (len(PRIMARY) + 1.2)
    ax.plot([NULL_RATE * tested_] * 2, [top - .5, top + len(PRIMARY) - .5], ls='--', color=MUTED, lw=1)
    ax.text(.995, top + 1.5, name.split(' ', 1)[1] + f'\n({int(null_calls.loc[null_calls.pool.eq(name), "label"].nunique())} splits)',
            va='center', ha='right', fontsize=7, color=INK, transform=ax.get_yaxis_transform())
ax.set_xscale('symlog', linthresh=1)
ax.set_xlim(-.3, 5e3)
ax.set_yticks(positions, labels_, fontsize=6.5)
ax.invert_yaxis()
ax.set_xlabel('pathways called in a same-species 2 vs 2 split (BH q ≤ 0.05)')
ax.set_title('a  Same-species null draws', loc='left', fontsize=9, fontweight='bold')

ax = fig.add_subplot(grid_[1, 0])
for strategy, marker in zip(PRIMARY, ['o', 's', '^', 'D']):
    rows = cross_draws[cross_draws.strategy.eq(strategy)]
    ax.scatter(rows.species_calls, rows.relabel_mean, s=10, marker=marker, facecolor='none',
               edgecolor=family_color[strategy], lw=.8, label=short[strategy])
upper = max(cross_draws.species_calls.max(), cross_draws.relabel_mean.max()) * 1.5 + 2
ax.plot([0, upper], [0, upper], color=MUTED, lw=.8, ls=':')
ax.set_xscale('symlog', linthresh=1), ax.set_yscale('symlog', linthresh=1)
ax.set_xlabel('species calls (2 human + 2 mouse)'), ax.set_ylabel('mean relabeled calls')
ax.legend(fontsize=6, frameon=False, loc='upper center', bbox_to_anchor=(.45, -.3), ncol=2, handletextpad=.2, columnspacing=.8)
ax.set_title('b  Cross-species draws (N5)', loc='left', fontsize=9, fontweight='bold')

ax = fig.add_subplot(grid_[1, 1])
kinds = ['offset', 'specimen × segment', 'within-unit (VIF)']
kind_style = {'offset': (AVERAGE, 'o'), 'specimen × segment': (POSITION, 's'), 'within-unit (VIF)': (MUTED, '^')}
pool_order = [*same_pools, 'N7 our cohort', 'N6 our AKI design']
for k, kind in enumerate(kinds):
    medians = [coherence_all.loc[coherence_all.pool.eq(p), kind].median() for p in pool_order]
    ax.scatter(medians, np.arange(len(pool_order)) + (k - 1) * .2, color=kind_style[kind][0],
               marker=kind_style[kind][1], s=16, label=kind)
ax.axvline(0, color=MUTED, lw=.8)
ax.set_yticks(range(len(pool_order)), [p.split(' ', 1)[1] for p in pool_order], fontsize=6.5)
ax.invert_yaxis()
ax.set_xlabel('median within-pathway correlation of member genes')
ax.legend(fontsize=6, frameon=False, loc='upper center', bbox_to_anchor=(.45, -.3), ncol=3, handletextpad=.2, columnspacing=.8)
ax.set_title('c  Coherence of specimen deviations', loc='left', fontsize=9, fontweight='bold')

ax = fig.add_subplot(grid_[2, 0])
order = [*PRIMARY, 'T_spatial joint (coordinate)']
relabel_cols = [c for c in aki_table if c.startswith('relabel_')]
for i, strategy in enumerate(order):
    ax.barh(i - .2, aki_table.loc[strategy, 'condition'], height=.36, color=family_color[strategy])
    ax.barh(i + .2, aki_table.loc[strategy, relabel_cols].mean(), height=.36, color=family_color[strategy], alpha=.35)
    ax.text(aki_table.loc[strategy, 'condition'] + 3, i - .2,
            f"{int(aki_table.loc[strategy, 'condition'])} vs {', '.join(str(int(v)) for v in aki_table.loc[strategy, relabel_cols])}",
            va='center', fontsize=6, color=INK)
ax.set_yticks(range(len(order)), [short[s] for s in order], fontsize=6.5)
ax.invert_yaxis()
ax.set_xlabel('pathways called (solid: AKI vs control; pale: mean of relabelings)')
ax.set_xlim(0, aki_table.loc[order, 'condition'].max() * 1.45 + 5)
ax.set_title('d  AKI positive control', loc='left', fontsize=9, fontweight='bold')

ax = fig.add_subplot(grid_[2, 1])
for marker_ in markers:
    ax.plot(range(len(regions)), region_delta.loc[marker_, regions], color=MUTED, lw=.6, alpha=.6)
ax.plot([], [], color=MUTED, lw=.6, label=f'each of {len(markers)} injury markers')
ax.plot(range(len(regions)), region_delta[regions].median(), color=INK, lw=2, marker='o', ms=4, label='median marker')
ax.axhline(0, color=MUTED, lw=.8)
ax.set_xticks(range(len(regions)), regions, fontsize=7)
ax.set_xlim(-.3, len(regions) - .7)
ax.set_ylabel('AKI − control, log expression')
ax.legend(fontsize=6, frameon=False, loc='upper left')
ax.set_title('e  Injury markers by region', loc='left', fontsize=9, fontweight='bold')
fig.savefig(figures / 'figure_R2_revision_many_draw_null.png', dpi=300, bbox_inches='tight')
fig.savefig(figures / 'figure_R2_revision_many_draw_null.pdf', bbox_inches='tight')
plt.show()

## 13 · Run record

In [ ]:
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'alpha': ALPHA,
          'pools': {name: {'donors': list(map(str, p['donors'])), 'tested_genes': int(len(p['genes'])),
                           'pathways': int(len(p['pathways']))} for name, p in pools.items()},
          'n_sets': N_SETS, 'n_cross_draws': N_CROSS, 'gseapy': gp.__version__,
          'R2_decision': R2_DECISION, 'D6_decision': 'pass' if D6_PASS else 'fail'}
(output / 'run_record.json').write_text(json.dumps(record, indent=2))
print(json.dumps({k: v for k, v in record.items() if k != 'pools'}, indent=2))